In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       4.2882, ∥Δ∥: 1.0907, η: 0.50, ∥Δᶜ∥: 0.5454, accepted: True , log 𝓛:  -3.3515
[  2] ∥∇∥:       0.5511, ∥Δ∥: 0.1392, η: 1.00, ∥Δᶜ∥: 0.1392, accepted: True , log 𝓛:  -3.3220
[  3] ∥∇∥:       0.0532, ∥Δ∥: 0.0150, η: 1.00, ∥Δᶜ∥: 0.0150, accepted: True , log 𝓛:  -3.3216
[  4] ∥∇∥:       0.0005, ∥Δ∥: 0.0001, η: 0.02, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  -3.3216

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 4, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([-0.1559, -0.5766]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-3.3515), tensor(-3.3220), tensor(-3.3216), tensor(-3.3216)]

In [3]:
torch.stack(history["loglik"])

tensor([-3.3515, -3.3220, -3.3216, -3.3216])

In [4]:
torch.stack(history["free_params"])

tensor([[-0.0357, -0.5442],
        [-0.1709, -0.5774],
        [-0.1559, -0.5766],
        [-0.1559, -0.5766]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([-0.1559, -0.5766]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([0.7321]), 'R/sigma^2': tensor([0.3156])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.0478, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321,
          0.0000],
         [0.0000, 1.0478, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000,
          0.7321],
         [0.7321, 0.0000, 1.0478, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321,
          0.0000],
         [0.0000, 0.7321, 0.0000, 1.0478, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000,
          0.7321],
         [0.7321, 0.0000, 0.7321, 0.0000, 1.0478, 0.0000, 0.7321, 0.0000, 0.7321,
          0.0000],
         [0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 1.0478, 0.0000, 0.7321, 0.0000,
          0.7321],
         [0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 1.0478, 0.0000, 0.7321,
          0.0000],
         [0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 1.0478, 0.0000,
          0.7321],
         [0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 1.0478,
          0.0000],
         [0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000, 0.7321, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.6561, -0.4716])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[0.0593, 0.0119],
        [0.0119, 0.1044]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.2436, 0.3231])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 0.9230, -0.6892])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[ 0.6599, -0.0067],
        [-0.0067,  0.6668]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([0.6599, 0.6668])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 0.9230, -0.6892])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[ 0.6599, -0.0067],
         [-0.0067,  0.6668]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([0.6599, 0.6668])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([-7.4236e-01, -2.7355e-01,  4.5222e-01, -1.5072e+00, -8.6841e-01,
         4.2749e-01,  1.2203e-01, -9.9789e-04, -2.4980e-01, -1.5936e-01])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 0.1806, -0.9628,  1.3752, -2.1964,  0.0546, -0.2617,  1.0450, -0.6902,
         0.6732, -0.8486])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 0.1806, -0.9628])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 0.1806, -0.9628])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 0.8993, -0.0089,  0.8618, -0.6759,  0.4293, -1.4035,  1.7114, -0.6919,
         1.1110, -0.9630])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([-0.0237,  0.6803, -0.0611,  0.0133, -0.4937, -0.7143,  0.7884, -0.0027,
         0.1880, -0.2738])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[0.0593, 0.0119],
         [0.0119, 0.1044]]),
 tensor([[ 0.0276,  0.0178],
         [ 0.0171, -0.0103]]),
 tensor([[ 0.0276,  0.0171],
         [ 0.0178, -0.0103]]),
 tensor([[0.0722, 0.0067],
         [0.0067, 0.0653]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-3.3216), 'cform')

In [25]:
reml.is_converged()

True